# Notebook 01: Data Acquisition
## EV Charging Gap Analysis — Texas Triangle Freight Corridor

This notebook downloads, filters, and saves all datasets needed for the analysis.

**What this notebook does:**
1. Downloads AFDC EV station data via API
2. Filters to Texas, splits DC Fast vs Level 2 chargers
3. Downloads Texas highway network (I-10, I-35, I-45) via `pygris`
4. Downloads Texas county boundaries via `pygris`
5. Loads OSM truck stop data from your manually downloaded GeoJSON
6. Saves all cleaned outputs to `data/processed/`

**Before running:**
- Paste your NREL API key into **Cell 2** below
- Make sure `data/raw/tx_truck_stops.geojson` is saved (from Phase 1, Step 2)

---
## Cell 1: Imports and Setup

> **What this does:** Loads all Python libraries and sets up the folder paths.

In [1]:
import pandas as pd
import geopandas as gpd
import pygris
import requests
import io
from shapely.geometry import Point
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('/Users/nadhirahendra/Documents/02-areas/QMSSGR5070-GIS/portfolio/post04-ev-charging-gap')
RAW       = BASE / 'data' / 'raw'
PROCESSED = BASE / 'data' / 'processed'

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Paths set up:')
print(f'  Raw data:       {RAW}')
print(f'  Processed data: {PROCESSED}')

Paths set up:
  Raw data:       /Users/nadhirahendra/Documents/02-areas/QMSSGR5070-GIS/portfolio/post04-ev-charging-gap/data/raw
  Processed data: /Users/nadhirahendra/Documents/02-areas/QMSSGR5070-GIS/portfolio/post04-ev-charging-gap/data/processed


---
## Cell 2: Download AFDC Station Data via API

> **What the AFDC API is:** The Alternative Fuels Data Center (AFDC), run by the US Department of Energy, provides a free API to query their full database of fuel stations. You registered at https://developer.nrel.gov and received an API key — that key is your credential to make requests.
>
> **What this cell does:** It calls the AFDC API filtered to Texas, open EV stations, and downloads the result as a CSV. The file is saved to `data/raw/afdc_stations_tx.csv` so the rest of the notebook can load it.
>
> **You only need to run this once.** If the file already exists, the cell skips the download automatically.

In [ ]:
# -------------------------------------------------------
# PASTE YOUR API KEY HERE
# Get it from: https://developer.nrel.gov/signup/
# -------------------------------------------------------
NREL_API_KEY = 'YOUR_API_KEY_HERE'
# -------------------------------------------------------

afdc_csv_path = RAW / 'afdc_stations_tx.csv'

# Delete the empty file from the previous run if it exists
if afdc_csv_path.exists():
    existing = pd.read_csv(afdc_csv_path, low_memory=False)
    if len(existing) == 0:
        afdc_csv_path.unlink()
        print('Removed empty CSV from previous run — re-downloading...')

if afdc_csv_path.exists():
    print(f'CSV already exists — skipping download.')
    print(f'  {afdc_csv_path}')
    print('Delete the file and re-run this cell if you want fresh data.')
else:
    print('Fetching Texas EV stations from AFDC API...')

    url = 'https://developer.nrel.gov/api/alt-fuel-stations/v1.csv'
    params = {
        'api_key':   NREL_API_KEY,
        'fuel_type': 'ELEC',
        'state':     'TX',
        # No status filter — we get all stations and filter to open ones in Python
        # No limit — defaults to returning all matching records
    }

    response = requests.get(url, params=params, timeout=60)

    if response.status_code == 200:
        with open(afdc_csv_path, 'wb') as f:
            f.write(response.content)
        preview = pd.read_csv(io.BytesIO(response.content), low_memory=False)
        print(f'Downloaded {len(preview):,} total stations (all statuses)')
        print(f'Saved to: {afdc_csv_path}')
    else:
        print(f'ERROR: API returned status {response.status_code}')
        print(response.text[:500])
        raise RuntimeError('AFDC API request failed — check your API key and try again.')

---
## Cell 3: Load the AFDC CSV

> **What this does:** Reads the CSV that Cell 2 downloaded. The file has dozens of columns — we keep only the ones we need: location, charger counts, network, and city.
>
> **Key column:** `EV DC Fast Count` tells us how many DC Fast charger ports a station has. A value of 0 or NaN means no DC Fast charging — that station is not useful for a commercial truck on a schedule.

In [3]:
afdc_raw = pd.read_csv(RAW / 'afdc_stations_tx.csv', low_memory=False)
print(f'Loaded {len(afdc_raw):,} rows')
print(f'Columns: {list(afdc_raw.columns[:15])}...')

Loaded 4,085 rows
Columns: ['Fuel Type Code', 'Station Name', 'Street Address', 'Intersection Directions', 'City', 'State', 'ZIP', 'Plus4', 'Station Phone', 'Status Code', 'Expected Date', 'Groups With Access Code', 'Access Days Time', 'Cards Accepted', 'BD Blends']...


---
## Cell 4: Filter and Split — DC Fast vs Level 2

> **Why we split:** For truck electrification, only DC Fast Chargers matter. A Level 2 charger takes 10-15 hours to fully charge an electric semi — unusable on a commercial schedule. DC Fast Chargers cut that to 1-2 hours. We keep both types in separate files so our maps can show the full picture.

In [ ]:
keep_cols = [
    'Station Name', 'City', 'State', 'ZIP',
    'Latitude', 'Longitude',
    'EV DC Fast Count', 'EV Level2 EVSE Num',
    'EV Connector Types', 'EV Network',
    'Access Code', 'Status Code'
]
keep_cols = [c for c in keep_cols if c in afdc_raw.columns]
afdc = afdc_raw[keep_cols].dropna(subset=['Latitude', 'Longitude']).copy()

# Filter to Texas only
if 'State' in afdc.columns:
    afdc = afdc[afdc['State'] == 'TX']

# Filter to open/available stations only
# Status Code 'E' = Available (open). 'P' = Planned, 'T' = Temporarily closed.
if 'Status Code' in afdc.columns:
    print(f'Status codes found: {afdc["Status Code"].value_counts().to_dict()}')
    afdc = afdc[afdc['Status Code'] == 'E']
    print(f'After filtering to open stations (Status Code = E): {len(afdc):,}')

if 'EV DC Fast Count' in afdc.columns:
    afdc['EV DC Fast Count'] = pd.to_numeric(afdc['EV DC Fast Count'], errors='coerce').fillna(0)
if 'EV Level2 EVSE Num' in afdc.columns:
    afdc['EV Level2 EVSE Num'] = pd.to_numeric(afdc['EV Level2 EVSE Num'], errors='coerce').fillna(0)

dc_fast = afdc[afdc['EV DC Fast Count'] > 0].copy()
level2  = afdc[afdc['EV Level2 EVSE Num'] > 0].copy()

print(f'Total open Texas EV stations: {len(afdc):,}')
print(f'DC Fast charger stations:     {len(dc_fast):,}')
print(f'Level 2 charger stations:     {len(level2):,}')

---
## Cell 5: Convert to GeoDataFrames

> **What is a GeoDataFrame?** A GeoDataFrame is like a regular pandas DataFrame, but each row has a geometry — here, a point built from Latitude and Longitude. CRS (`EPSG:4326`) is the standard lat/lon coordinate system (WGS84).

In [5]:
def df_to_gdf(df, lat_col='Latitude', lon_col='Longitude', crs='EPSG:4326'):
    geometry = [Point(lon, lat) for lon, lat in zip(df[lon_col], df[lat_col])]
    return gpd.GeoDataFrame(df, geometry=geometry, crs=crs)

dc_fast_gdf = df_to_gdf(dc_fast)
level2_gdf  = df_to_gdf(level2)

print(f'DC Fast GeoDataFrame: {len(dc_fast_gdf):,} stations, CRS = {dc_fast_gdf.crs}')
print(f'Level 2 GeoDataFrame: {len(level2_gdf):,} stations, CRS = {level2_gdf.crs}')
print()
print('DC Fast sample:')
print(dc_fast_gdf[['Station Name', 'City', 'EV DC Fast Count', 'geometry']].head(5).to_string())
dc_fast_gdf.head()

DC Fast GeoDataFrame: 836 stations, CRS = EPSG:4326
Level 2 GeoDataFrame: 3,183 stations, CRS = EPSG:4326

DC Fast sample:
             Station Name        City  EV DC Fast Count                    geometry
1           Grubbs Nissan     Bedford               2.0  POINT (-97.16073 32.83945)
4     Nissan - Fort Worth  Fort Worth               1.0  POINT (-97.47845 32.72102)
5      Baker Nissan North     Houston               1.0   POINT (-95.6124 29.91354)
6  Central Houston Nissan     Houston               4.0  POINT (-95.42258 29.67646)
7          McDavid Nissan     Houston               1.0  POINT (-95.22531 29.62556)


,Station Name,City,State,ZIP,Latitude,Longitude,EV DC Fast Count,EV Level2 EVSE Num,EV Connector Types,EV Network,Access Code,Status Code,Maximum Vehicle Class,Facility Type,vehicle_class,geometry
1,Grubbs Nissan,Bedford,TX,76022,32.839448,-97.160732,2.0,1.0,CHADEMO J1772 J1772COMBO,Non-Networked,public,E,LD,CAR_DEALER,LD,POINT (-97.16073 32.83945)
4,Nissan - Fort Worth,Fort Worth,TX,76116,32.721017,-97.478446,1.0,1.0,CHADEMO J1772 J1772COMBO,Non-Networked,public,E,LD,CAR_DEALER,LD,POINT (-97.47845 32.72102)
5,Baker Nissan North,Houston,TX,77065,29.913542,-95.612404,1.0,3.0,CHADEMO J1772 J1772COMBO,Non-Networked,public,E,LD,CAR_DEALER,LD,POINT (-95.6124 29.91354)
6,Central Houston Nissan,Houston,TX,77054,29.676460,-95.422580,4.0,1.0,CHADEMO J1772 J1772COMBO,Non-Networked,public,E,LD,CAR_DEALER,LD,POINT (-95.42258 29.67646)
7,McDavid Nissan,Houston,TX,77034,29.625565,-95.225310,1.0,1.0,CHADEMO J1772,Non-Networked,public,E,LD,CAR_DEALER,LD,POINT (-95.22531 29.62556)


---
## Cell 6: Download Texas Highway Network

> **What is `pygris`?** A Python library that downloads US Census TIGER/Line geographic data directly — no manual shapefile downloading needed.
>
> **Why these three routes?** The Texas Triangle is I-10 (Houston to San Antonio), I-35 (San Antonio to Dallas), and I-45 (Dallas to Houston). These three interstates carry the majority of intercity truck traffic in Texas.

In [6]:
print('Downloading national primary roads from Census TIGER (this takes ~30 seconds)...')
# primary_roads() downloads the full national layer — no state argument
all_roads = pygris.primary_roads(year=2022)
print(f'Downloaded {len(all_roads):,} total road segments, CRS: {all_roads.crs}')

# Filter to Texas by bounding box (lon -106.65 to -93.51, lat 25.84 to 36.50)
tx_roads = all_roads.cx[-106.65:-93.51, 25.84:36.50].copy()
print(f'Road segments within Texas bounding box: {len(tx_roads):,}')

# Filter to the Texas Triangle interstates
triangle_hwys = tx_roads[
    tx_roads['FULLNAME'].str.contains(r'I-\s*(10|35|45)\b', regex=True, na=False)
].copy()

print(f'Triangle highway segments: {len(triangle_hwys):,}')
print(f'Unique names: {sorted(triangle_hwys["FULLNAME"].unique())}')

Downloaded 17,389 total road segments, CRS: EPSG:4269
Road segments within Texas bounding box: 2,218
Triangle highway segments: 215
Unique names: ['E I- 10', 'I- 10', 'I- 10 (Hov)', 'I- 10 W', 'I- 35', 'I- 35 E', 'I- 35 N', 'I- 35 W', 'I- 45', 'I- 45 Hov', 'I- 45 S', 'I- 45-Hov', 'N I-35', 'S I- 35', 'S I-35']


---
## Cell 7: Download Texas County Boundaries

> **Why counties?** County boundaries are used as a background reference layer on maps — they help readers orient themselves geographically.

In [7]:
print('Downloading Texas county boundaries...')
tx_counties = pygris.counties(state='TX', year=2022)
print(f'Downloaded {len(tx_counties):,} counties, CRS = {tx_counties.crs}')

Using FIPS code '48' for input 'TX'
Downloaded 254 counties, CRS = EPSG:4269


---
## Cell 8: Load Truck Stop Data

> **What we are loading:** The GeoJSON you downloaded from OpenStreetMap via Overpass Turbo. It contains highway-grade fuel stops tagged `hgv=yes` or `truck=yes`. These are where truck drivers currently stop — but most have diesel pumps only, no DC Fast chargers.

In [8]:
import json
import time

truck_geojson_path = RAW / 'tx_truck_stops.geojson'

if truck_geojson_path.exists():
    print('tx_truck_stops.geojson already exists — skipping download.')
else:
    print('Downloading Texas truck stops from Overpass API...')

    # Bounding box covers the Texas Triangle corridor (not all of Texas — faster query)
    # lon_min, lat_min, lon_max, lat_max
    BBOX = '29.0,-99.5,33.5,-93.5'

    query = f"""
[out:json][timeout:60];
(
  node["amenity"="fuel"]["hgv"="yes"]({BBOX});
  node["amenity"="fuel"]["truck"~"yes|designated"]({BBOX});
  node["highway"="rest_area"]({BBOX});
  node["highway"="services"]({BBOX});
  way["amenity"="fuel"]["hgv"="yes"]({BBOX});
  way["amenity"="fuel"]["truck"~"yes|designated"]({BBOX});
);
out center;
"""

    # Try multiple Overpass servers in case one is busy
    servers = [
        'https://overpass-api.de/api/interpreter',
        'https://overpass.kumi.systems/api/interpreter',
        'https://overpass.openstreetmap.fr/api/interpreter',
    ]

    response = None
    for server in servers:
        try:
            print(f'  Trying {server}...')
            r = requests.post(server, data={'data': query}, timeout=90)
            if r.status_code == 200 and 'elements' in r.json():
                response = r
                print(f'  Success.')
                break
            else:
                print(f'  Failed (status {r.status_code}), trying next server...')
        except Exception as e:
            print(f'  Error: {e}, trying next server...')
        time.sleep(2)

    if response is None:
        raise RuntimeError('All Overpass servers failed. Try again in a few minutes.')

    data = response.json()
    elements = data.get('elements', [])
    print(f'Retrieved {len(elements):,} elements from Overpass')

    # Convert OSM elements to GeoJSON features
    features = []
    for el in elements:
        # Nodes have lat/lon directly; ways have center.lat/center.lon
        if el['type'] == 'node':
            lat, lon = el.get('lat'), el.get('lon')
        elif el['type'] == 'way' and 'center' in el:
            lat, lon = el['center']['lat'], el['center']['lon']
        else:
            continue

        if lat is None or lon is None:
            continue

        features.append({
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [lon, lat]},
            'properties': el.get('tags', {})
        })

    geojson = {'type': 'FeatureCollection', 'features': features}

    with open(truck_geojson_path, 'w') as f:
        json.dump(geojson, f)

    print(f'Saved {len(features):,} truck stop points to {truck_geojson_path}')

tx_truck_stops.geojson already exists — skipping download.


In [9]:
truck_path = RAW / 'tx_truck_stops.geojson'

if not truck_path.exists():
    print('WARNING: tx_truck_stops.geojson not found.')
    print('  Follow Phase 1 Step 2 to download it from Overpass Turbo.')
    print('  Creating an empty placeholder so the rest of the notebook can run.')
    import geopandas as gpd
    from shapely.geometry import Point
    truck_stops = gpd.GeoDataFrame({'geometry': []}, crs='EPSG:4326')
else:
    truck_stops = gpd.read_file(truck_path)
    print(f'Loaded {len(truck_stops):,} truck stop features, CRS: {truck_stops.crs}')
    print(f'Geometry types: {truck_stops.geometry.geom_type.value_counts().to_dict()}')

    # Convert any non-point geometries to centroids
    if len(truck_stops) > 0 and truck_stops.geometry.geom_type.isin(['Polygon', 'MultiPolygon', 'LineString']).any():
        truck_stops['geometry'] = truck_stops['geometry'].centroid
        print('Converted non-point geometries to centroids')

    truck_stops = truck_stops[truck_stops.geometry.geom_type == 'Point'].copy()
    print(f'Final truck stop count: {len(truck_stops):,}')

Loaded 194 truck stop features, CRS: EPSG:4326
Geometry types: {'Point': 194}
Final truck stop count: 194


---
## Cell 9: Align Coordinate Reference Systems

> **Why this matters:** All layers must share the same CRS before you can compare or overlay them. Census TIGER uses `EPSG:4269` (NAD83) and AFDC uses `EPSG:4326` (WGS84). We standardize everything to `EPSG:4326` for storage. Distance calculations in Notebook 02 will use UTM instead.

In [10]:
TARGET_CRS = 'EPSG:4326'

triangle_hwys = triangle_hwys.to_crs(TARGET_CRS)
tx_counties   = tx_counties.to_crs(TARGET_CRS)

if truck_stops.crs is None:
    truck_stops = truck_stops.set_crs(TARGET_CRS)
else:
    truck_stops = truck_stops.to_crs(TARGET_CRS)

print('CRS alignment complete:')
print(f'  DC Fast chargers:  {dc_fast_gdf.crs}')
print(f'  Level 2 chargers:  {level2_gdf.crs}')
print(f'  Triangle highways: {triangle_hwys.crs}')
print(f'  TX counties:       {tx_counties.crs}')
print(f'  Truck stops:       {truck_stops.crs}')

CRS alignment complete:
  DC Fast chargers:  EPSG:4326
  Level 2 chargers:  EPSG:4326
  Triangle highways: EPSG:4326
  TX counties:       EPSG:4326
  Truck stops:       EPSG:4326


---
## Cell 10: Summary Check

> **Sanity check before saving.** Texas has a growing but still sparse EV network — a few hundred DC Fast stations statewide is typical as of 2025-2026. If the DC Fast count is zero, check that `EV DC Fast Count` exists as a column name in your CSV.

In [11]:
print('=' * 50)
print('DATASET SUMMARY')
print('=' * 50)
print(f'Total TX EV stations loaded:    {len(afdc):,}')
print(f'  DC Fast charger stations:     {len(dc_fast_gdf):,}')
print(f'  Level 2 charger stations:     {len(level2_gdf):,}')
print(f'Triangle highway segments:      {len(triangle_hwys):,}')
print(f'Texas counties:                 {len(tx_counties):,}')
print(f'Truck stops (OSM):              {len(truck_stops):,}')
print()
print('Bounding boxes (should be within TX: lon -106 to -94, lat 26 to 36):')
print(f'  DC Fast: {dc_fast_gdf.total_bounds.round(2)}')
print(f'  Highways: {triangle_hwys.total_bounds.round(2)}')

DATASET SUMMARY
Total TX EV stations loaded:    3,966
  DC Fast charger stations:     836
  Level 2 charger stations:     3,183
Triangle highway segments:      215
Texas counties:                 254
Truck stops (OSM):              194

Bounding boxes (should be within TX: lon -106 to -94, lat 26 to 36):
  DC Fast: [-106.58   25.94  -93.82   35.54]
  Highways: [-107.3    27.51  -93.     36.59]


---
## Cell 11: Save All Processed Outputs

> **Why GeoJSON?** GeoJSON is a plain-text open format that works in Python, QGIS, ArcGIS Online, and web browsers. It avoids the 10-character field name limit that shapefiles impose.

In [ ]:
outputs = {
    'dc_fast_tx.geojson':    dc_fast_gdf,
    'level2_tx.geojson':     level2_gdf,
    'triangle_hwys.geojson': triangle_hwys,
    'tx_counties.geojson':   tx_counties,
    'truck_stops.geojson':   truck_stops,
}

for filename, gdf in outputs.items():
    out_path = PROCESSED / filename
    gdf.to_file(out_path, driver='GeoJSON')
    print(f'Saved: {filename} ({len(gdf):,} features)')

print()
print('All files saved to data/processed/')

---
## Final Verification

Run this cell to confirm all five output files exist and are non-empty.

In [13]:
expected_files = [
    'dc_fast_tx.geojson',
    'level2_tx.geojson',
    'triangle_hwys.geojson',
    'tx_counties.geojson',
    'truck_stops.geojson',
]

print('Verification:')
all_good = True
for fname in expected_files:
    fpath = PROCESSED / fname
    exists = fpath.exists()
    size_kb = fpath.stat().st_size / 1024 if exists else 0
    status = 'OK' if (exists and size_kb > 1) else 'MISSING or EMPTY'
    if status != 'OK':
        all_good = False
    print(f'  [{status}] {fname} ({size_kb:.0f} KB)')

print()
if all_good:
    print('All files verified. Ready for Phase 2: Spatial Analysis.')
else:
    print('Some files are missing. Check the cells above for errors.')

Verification:
  [OK] dc_fast_tx.geojson (418 KB)
  [OK] level2_tx.geojson (1488 KB)
  [OK] triangle_hwys.geojson (2550 KB)
  [OK] tx_counties.geojson (26487 KB)
  [OK] truck_stops.geojson (214 KB)

All files verified. Ready for Phase 2: Spatial Analysis.
